# 💻 Developer Survey 2020–2026 — EDA & Analysis

**12K responses · 50 countries · 7 years · Skills + Salary + AI**

1. Overview
2. Language & Framework Popularity
3. Salary Analysis
4. AI Adoption & Impact
5. Job Satisfaction & Burnout
6. Work Mode & Remote Trends
7. Demographics & Diversity
8. Experience vs. Salary
9. Technology Correlation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
print("✅ Ready")

## 1. Load & Overview

In [ ]:
INPUT = "/kaggle/input/developer-survey-2020-2026-skills-salary-ai"

df = pd.read_csv(f"{INPUT}/developer_survey.csv")
lang_trends = pd.read_csv(f"{INPUT}/language_trends.csv")
salary_rc = pd.read_csv(f"{INPUT}/salary_by_role_country.csv")
ai_trends = pd.read_csv(f"{INPUT}/ai_adoption_trends.csv")

print(f"Shape: {df.shape}")
print(f"Countries: {df['country'].nunique()} | Years: {sorted(df['survey_year'].unique())}")
print(f"Avg salary (employed): ${df[df['annual_salary_usd']>0]['annual_salary_usd'].mean():,.0f}")
print(f"Missing values: {df.isnull().sum().sum()}")

In [ ]:
df.head()

## 2. Language & Framework Popularity

In [ ]:
# All-time language popularity
all_langs = df['languages_used'].str.split(';').explode()
lang_counts = all_langs.value_counts().head(15)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

lang_counts.sort_values().plot.barh(ax=axes[0], color=sns.color_palette("viridis", len(lang_counts)), edgecolor='black')
axes[0].set_title('Top 15 Programming Languages', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Users')

# Language trends over time
top_langs = ['Python', 'JavaScript', 'TypeScript', 'Rust', 'Go', 'Java']
for lang in top_langs:
    lt = lang_trends[lang_trends['language'] == lang].sort_values('year')
    axes[1].plot(lt['year'], lt['pct_respondents'], marker='o', label=lang, linewidth=2)
axes[1].set_title('Language Trends Over Time', fontsize=14, fontweight='bold')
axes[1].set_ylabel('% of Respondents')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Frameworks
all_fw = df['frameworks_used'].str.split(';').explode()
fw_counts = all_fw.value_counts().head(15)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

fw_counts.sort_values().plot.barh(ax=axes[0], color=sns.color_palette("magma", len(fw_counts)), edgecolor='black')
axes[0].set_title('Top 15 Frameworks & Libraries', fontsize=14, fontweight='bold')

# Databases
all_db = df['databases_used'].str.split(';').explode()
db_counts = all_db.value_counts().head(12)
db_counts.sort_values().plot.barh(ax=axes[1], color=sns.color_palette("crest", len(db_counts)), edgecolor='black')
axes[1].set_title('Top 12 Databases', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Salary Analysis

In [ ]:
employed = df[df['annual_salary_usd'] > 0]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

role_salary = employed.groupby('role')['annual_salary_usd'].median().sort_values()
role_salary.plot.barh(ax=axes[0], color='#3498db', edgecolor='black')
axes[0].set_title('Median Salary by Role (USD)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Annual Salary (USD)')

top_countries = employed.groupby('country')['respondent_id'].count().nlargest(12).index
country_salary = employed[employed['country'].isin(top_countries)].groupby('country')['annual_salary_usd'].median().sort_values()
country_salary.plot.barh(ax=axes[1], color='#2ecc71', edgecolor='black')
axes[1].set_title('Median Salary by Country (Top 12)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Annual Salary (USD)')

plt.tight_layout()
plt.show()

In [ ]:
# Salary distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

employed['annual_salary_usd'].clip(upper=250000).plot.hist(bins=50, ax=axes[0], color='#9b59b6', edgecolor='black', alpha=0.8)
axes[0].set_title('Salary Distribution (capped at $250K)', fontweight='bold')
axes[0].set_xlabel('USD')
axes[0].axvline(employed['annual_salary_usd'].median(), color='red', linestyle='--', label=f"Median: ${employed['annual_salary_usd'].median():,.0f}")
axes[0].legend()

# By education
ed_order = ['High school', 'Some college', 'Associate degree', 'Self-taught / Bootcamp', "Bachelor's degree", "Master's degree", 'Doctorate (PhD)']
ed_salary = employed.groupby('education_level')['annual_salary_usd'].median().reindex(ed_order).dropna()
ed_salary.plot.barh(ax=axes[1], color='#e67e22', edgecolor='black')
axes[1].set_title('Median Salary by Education', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. AI Adoption & Impact

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# AI usage frequency
usage_order = ['Multiple times daily', 'Daily', 'Weekly', 'Monthly', 'Rarely', 'Never']
usage_counts = df['ai_usage_frequency'].value_counts().reindex(usage_order)
usage_counts.plot.bar(ax=axes[0,0], color=sns.color_palette("Blues_r", len(usage_order)), edgecolor='black')
axes[0,0].set_title('AI Tool Usage Frequency', fontsize=13, fontweight='bold')
axes[0,0].tick_params(axis='x', rotation=30)

# AI tools
all_ai = df['ai_tools_used'].str.split(';').explode()
ai_counts = all_ai.value_counts().head(10)
ai_counts.sort_values().plot.barh(ax=axes[0,1], color='#e74c3c', edgecolor='black')
axes[0,1].set_title('Top 10 AI Tools', fontsize=13, fontweight='bold')

# AI sentiment
sent_order = ['Very positive', 'Somewhat positive', 'Neutral', 'Somewhat negative', 'Very negative']
sent_counts = df['ai_sentiment'].value_counts().reindex(sent_order)
sent_counts.plot.bar(ax=axes[1,0], color=['#2ecc71','#27ae60','#f39c12','#e67e22','#e74c3c'], edgecolor='black')
axes[1,0].set_title('AI Sentiment', fontsize=13, fontweight='bold')
axes[1,0].tick_params(axis='x', rotation=30)

# AI adoption over years
axes[1,1].plot(ai_trends['survey_year'], ai_trends['pct_daily_ai'], marker='o', label='Daily AI Users (%)', color='#3498db', linewidth=2)
axes[1,1].plot(ai_trends['survey_year'], ai_trends['pct_positive_ai'], marker='s', label='Positive Sentiment (%)', color='#2ecc71', linewidth=2)
axes[1,1].plot(ai_trends['survey_year'], ai_trends['pct_threat_perception'], marker='^', label='Job Threat (%)', color='#e74c3c', linewidth=2)
axes[1,1].set_title('AI Trends Over Time', fontsize=13, fontweight='bold')
axes[1,1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# AI productivity boost by usage frequency
fig, ax = plt.subplots(figsize=(10, 5))
usage_prod = df.groupby('ai_usage_frequency')['ai_productivity_boost_pct'].mean().reindex(usage_order)
usage_prod.plot.bar(ax=ax, color=sns.color_palette("Greens_r", len(usage_order)), edgecolor='black')
ax.set_title('Self-Reported AI Productivity Boost by Usage Frequency', fontsize=14, fontweight='bold')
ax.set_ylabel('Avg Productivity Boost (%)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 5. Job Satisfaction & Burnout

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df['job_satisfaction'].value_counts().sort_index().plot.bar(ax=axes[0], color=sns.color_palette("RdYlGn", 5), edgecolor='black')
axes[0].set_title('Job Satisfaction (1-5)', fontweight='bold')
axes[0].set_xlabel('Rating')

burnout_order = ['Never', 'Rarely', 'Sometimes', 'Often', 'Always']
df['burnout_frequency'].value_counts().reindex(burnout_order).plot.bar(ax=axes[1], color=['#2ecc71','#27ae60','#f39c12','#e67e22','#e74c3c'], edgecolor='black')
axes[1].set_title('Burnout Frequency', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)

# Burnout by work mode
burnout_wm = pd.crosstab(df['work_mode'], df['burnout_frequency'], normalize='index')[burnout_order] * 100
burnout_wm.plot.bar(ax=axes[2], stacked=True, color=['#2ecc71','#27ae60','#f39c12','#e67e22','#e74c3c'], edgecolor='black')
axes[2].set_title('Burnout by Work Mode', fontweight='bold')
axes[2].set_ylabel('%')
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(bbox_to_anchor=(1.05, 1), fontsize=8)

plt.tight_layout()
plt.show()

## 6. Work Mode & Remote Trends

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

wm_counts = df['work_mode'].value_counts()
wm_counts.plot.pie(ax=axes[0], autopct='%1.1f%%', colors=['#3498db','#2ecc71','#e67e22'], textprops={'fontsize': 12})
axes[0].set_title('Work Mode Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('')

# Remote preference
df['remote_work_preference_1to5'].value_counts().sort_index().plot.bar(ax=axes[1], color=sns.color_palette("Blues", 5), edgecolor='black')
axes[1].set_title('Remote Work Preference (1=Office, 5=Fully Remote)', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Demographics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

df['age'].plot.hist(bins=30, ax=axes[0,0], color='#3498db', edgecolor='black')
axes[0,0].set_title('Age Distribution', fontweight='bold')

df['gender'].value_counts().plot.bar(ax=axes[0,1], color=['#3498db','#e74c3c','#9b59b6','#95a5a6'], edgecolor='black')
axes[0,1].set_title('Gender Distribution', fontweight='bold')
axes[0,1].tick_params(axis='x', rotation=30)

df['education_level'].value_counts().plot.barh(ax=axes[1,0], color='#e67e22', edgecolor='black')
axes[1,0].set_title('Education Level', fontweight='bold')

df['primary_os'].value_counts().plot.pie(ax=axes[1,1], autopct='%1.1f%%', colors=['#3498db','#2ecc71','#e67e22'], textprops={'fontsize':12})
axes[1,1].set_title('Primary OS', fontweight='bold')
axes[1,1].set_ylabel('')

plt.tight_layout()
plt.show()

## 8. Experience vs. Salary

In [ ]:
employed = df[df['annual_salary_usd'] > 0]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter
axes[0].scatter(employed['years_of_experience'], employed['annual_salary_usd'].clip(upper=300000), alpha=0.1, s=8, c='#3498db')
axes[0].set_title('Experience vs Salary', fontweight='bold')
axes[0].set_xlabel('Years of Experience')
axes[0].set_ylabel('Annual Salary (USD)')

# Binned
exp_bins = pd.cut(employed['years_of_experience'], bins=[0,2,5,10,15,20,40], labels=['0-2','3-5','6-10','11-15','16-20','20+'])
exp_salary = employed.groupby(exp_bins)['annual_salary_usd'].median()
exp_salary.plot.bar(ax=axes[1], color='#2ecc71', edgecolor='black')
axes[1].set_title('Median Salary by Experience Band', fontweight='bold')
axes[1].set_ylabel('Median Salary (USD)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 9. Technology Correlation

In [ ]:
# IDE / Editor popularity
all_ide = df['ide_editor'].str.split(';').explode()
ide_counts = all_ide.value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ide_counts.sort_values().plot.barh(ax=axes[0], color='#9b59b6', edgecolor='black')
axes[0].set_title('IDE / Editor Popularity', fontweight='bold')

# Cloud platforms
all_cloud = df['cloud_platforms_used'].str.split(';').explode()
cloud_counts = all_cloud.value_counts()
cloud_counts.sort_values().plot.barh(ax=axes[1], color='#e67e22', edgecolor='black')
axes[1].set_title('Cloud Platform Popularity', fontweight='bold')

plt.tight_layout()
plt.show()

## 📋 Key Takeaways

- **JavaScript remains king** but **Python and TypeScript** are closing fast
- **Rust and Go** show consistent year-over-year growth
- **VS Code** dominates IDEs at 72%, with **Cursor** rising as the AI-native editor
- **53% of developers** use AI tools daily or multiple times daily
- **ChatGPT** is the most used AI tool (62%), followed by **GitHub Copilot** (45%) and **Claude** (28%)
- **ML Engineers and Engineering Managers** earn the highest median salaries
- **Remote work** preference is strong — 75% rate it 4 or 5 out of 5
- **33% of developers** report burnout "Often" or "Always"
- **Self-taught developers** earn competitive salaries compared to degree holders

**Next steps:**
- Salary prediction model (XGBoost)
- AI adoption clustering
- Burnout risk classifier
- Tech stack recommender system

---
*If this was useful, please upvote! 🙏*